# rag-agent-failure-catalog: quickstart

Three failures, end to end: **reproduce**, **detect**, **mitigate**. Everything here runs offline with no API keys.

The offline stubs *model* each failure so the detectors and mitigations can be exercised deterministically. To measure a real model, run the entry's `reproduce.py` with `--live` (see the README).

In [ ]:
import sys
from pathlib import Path

# This notebook lives in examples/. Put the repo root on the path.
sys.path.insert(0, str(Path.cwd().parent))

from common.utils import load_catalog_module
from common.retrieval_simulator import KeywordRetriever

## 1. Ungrounded answer

The user asks about a free trial. The docs do not mention one. The retriever still returns its nearest chunks, and the (stubbed) model answers anyway.

In [ ]:
rep = load_catalog_module("01_ungrounded_answer", "reproduce")
det = load_catalog_module("01_ungrounded_answer", "detect")
mit = load_catalog_module("01_ungrounded_answer", "mitigate")

client = rep.get_client(rep.simulated_model)
retriever = KeywordRetriever()

answer, context = rep.naive_rag_answer(client, rep.QUESTION, retriever)
print("ANSWER:", answer, "\n")

for f in det.find_unsupported(answer, context):
    print(f"flagged={f.flagged!s:<5} support={f.score:.2f}  {f.sentence}")

In [ ]:
result = mit.guarded_answer(client, rep.QUESTION, retriever)
print("grounded:", result.grounded)
print("reason  :", result.reason)
print("returned:", result.text)

## 3. Retrieval that looks fine but misses intent

Every query returns `k` results and the top chunk shares a keyword with the question. Surface metrics are green. The intent metric is not.

In [ ]:
rep3 = load_catalog_module("03_retrieval_surface_metrics", "reproduce")
mit3 = load_catalog_module("03_retrieval_surface_metrics", "mitigate")

r = KeywordRetriever()
rows = rep3.evaluate(r, k=3)
rep3.print_table(rows)
print(rep3.summarize(rows))

In [ ]:
rows_after = rep3.evaluate(r, k=3, rewrite=mit3.expand_query)
print(rep3.summarize(rows_after))

## 9. Silent tool failure

The order service is down. The wrapper swallows the exception and returns `[]`. The agent reports that a customer with two open orders has none.

In [ ]:
from common.agent_sim import run_agent

rep9 = load_catalog_module("09_silent_tool_failure", "reproduce")
det9 = load_catalog_module("09_silent_tool_failure", "detect")
mit9 = load_catalog_module("09_silent_tool_failure", "mitigate")

service = rep9.OrderService(down=True)
bad_tool = rep9.make_swallowing_tool(service)

trace = run_agent(rep9.TASK, rep9.agent_policy, {"lookup_orders": bad_tool}, verbose=False)
print("agent says:", trace.final_answer)
print("canary passes?", det9.canary(bad_tool, {"customer_id": "C-1042"}))
print("audit:", [f.kind for f in det9.audit_trace(trace)])

In [ ]:
def lookup_orders(customer_id):
    return service.lookup(customer_id)

good_tools = {"lookup_orders": mit9.typed_tool(lookup_orders)}
trace = run_agent(rep9.TASK, rep9.agent_policy, good_tools, verbose=False, final_hook=mit9.FailureAwareGate())
print("agent says:", trace.final_answer)

## Where to go next

- Run `python catalog/07_context_pollution/reproduce.py --live` to measure position sensitivity on your own model.
- Add your own eval cases to `catalog/03_retrieval_surface_metrics/reproduce.py`, drawn from real queries.
- Read [CONTRIBUTING.md](../CONTRIBUTING.md) to add a failure mode.